# 00_create_sample_dataset_for_embedding

A) Create sample exactly 1000 lines from each month while maintaining the compressed format, you can use the following script.
This code uses np.random.choice to pick random indices and then applies that same mask across all arrays within the .npz file to ensure the data stays aligned (i.e., the ID still matches the summary).

B) Section B we remove core public sector jobs from Onet v30.0 reducing from 894 to 861 (drop 33 core public sector jobs).

In [ ]:
# lets transfer from previous generated and add domain to role_text in order to enrich it;

In [3]:
se for rodar tem rodar tudo de novo e apague todos os npz do stage2!
#!rm -f /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/**/*.npz


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import time
from datetime import datetime

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
start_time = time.time()

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")
output_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet")

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

all_data = []

for month in months:
    npz_file = src_root / f"{month}.npz"
    
    if not npz_file.exists():
        print(f"⚠️  Skipping {month} - file not found")
        continue
    
    # Load NPZ and extract job_ids
    with np.load(npz_file, allow_pickle=True) as data:
        job_ids = data["job_ids"].astype(str)
        
        # Create dataframe for this month
        df_month = pd.DataFrame({
            'id': job_ids,
            'npz_file': f"{month}.npz"
        })
        
        all_data.append(df_month)
        print(f"✓ {month}.npz: {len(job_ids):,} IDs")

# Concatenate all months
print(f"\nConcatenating {len(all_data)} files...")
df_combined = pd.concat(all_data, ignore_index=True)

print(f"Total rows before deduplication: {len(df_combined):,}")

# Sort by npz_file then drop duplicates by id
df_combined = df_combined.sort_values('npz_file').reset_index(drop=True)
rows_before = len(df_combined)
df_deduplicated = df_combined.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

duplicates_removed = rows_before - len(df_deduplicated)
print(f"Duplicates removed: {duplicates_removed:,}")
print(f"Final rows: {len(df_deduplicated):,}")

# Save
output_file.parent.mkdir(parents=True, exist_ok=True)
df_deduplicated.to_parquet(output_file, index=False)

duration = time.time() - start_time
print(f"\nSaved to: {output_file}")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duration: {duration:.2f}s ({duration/60:.2f}m)")

Started at: 2026-02-16 21:37:18

✓ adzuna_month01.npz: 2,657,586 IDs
✓ adzuna_month02.npz: 2,229,436 IDs
✓ adzuna_month03.npz: 2,303,805 IDs
✓ adzuna_month04.npz: 2,331,704 IDs
✓ adzuna_month05.npz: 2,546,484 IDs
✓ adzuna_month06.npz: 1,992,787 IDs
✓ adzuna_month07.npz: 2,183,287 IDs
✓ adzuna_month08.npz: 1,913,920 IDs
✓ adzuna_month09.npz: 1,860,287 IDs
✓ adzuna_month10.npz: 2,179,371 IDs
✓ adzuna_month11.npz: 2,040,587 IDs
✓ adzuna_month12.npz: 1,739,463 IDs
✓ adzuna_month13.npz: 2,537,947 IDs
✓ adzuna_month14.npz: 21,923 IDs

Concatenating 14 files...
Total rows before deduplication: 28,538,587
Duplicates removed: 8,857,090
Final rows: 19,681,497

Saved to: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet
Completed at: 2026-02-16 21:37:56
Duration: 38.06s (0.63m)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import time
from datetime import datetime

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
start_time = time.time()

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")
parquet_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet")

# Load the deduplicated IDs
df_deduplicated = pd.read_parquet(parquet_file)
print(f"Loaded deduplicated parquet: {len(df_deduplicated):,} unique IDs\n")

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

total_original = 0
total_filtered = 0

for month in months:
    npz_file = src_root / f"{month}.npz"
    
    if not npz_file.exists():
        print(f"⚠️  Skipping {month} - file not found")
        continue
    
    # Load NPZ
    with np.load(npz_file, allow_pickle=True) as data:
        job_ids_original = data["job_ids"].astype(str)
        n_original = len(job_ids_original)
        
        # Get filtered IDs for this month
        filtered_ids = df_deduplicated[df_deduplicated['npz_file'] == f"{month}.npz"]['id'].values
        n_filtered = len(filtered_ids)
        
        # Calculate reduction
        reduction = n_original - n_filtered
        reduction_pct = (reduction / n_original * 100) if n_original > 0 else 0
        
        total_original += n_original
        total_filtered += n_filtered
        
        print(f"{month}.npz:")
        print(f"  Original: {n_original:,}")
        print(f"  Filtered: {n_filtered:,}")
        print(f"  Reduction: {reduction:,} ({reduction_pct:.2f}%)\n")

# Overall stats
total_reduction = total_original - total_filtered
total_reduction_pct = (total_reduction / total_original * 100) if total_original > 0 else 0

print("=" * 50)
print(f"TOTAL ACROSS ALL MONTHS:")
print(f"  Original: {total_original:,}")
print(f"  Filtered: {total_filtered:,}")
print(f"  Reduction: {total_reduction:,} ({total_reduction_pct:.2f}%)")

duration = time.time() - start_time
print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duration: {duration:.2f}s ({duration/60:.2f}m)")

Started at: 2026-02-16 21:46:56

Loaded deduplicated parquet: 19,681,497 unique IDs

adzuna_month01.npz:
  Original: 2,657,586
  Filtered: 2,657,586
  Reduction: 0 (0.00%)

adzuna_month02.npz:
  Original: 2,229,436
  Filtered: 1,539,063
  Reduction: 690,373 (30.97%)

adzuna_month03.npz:
  Original: 2,303,805
  Filtered: 1,684,605
  Reduction: 619,200 (26.88%)

adzuna_month04.npz:
  Original: 2,331,704
  Filtered: 1,500,944
  Reduction: 830,760 (35.63%)

adzuna_month05.npz:
  Original: 2,546,484
  Filtered: 1,845,262
  Reduction: 701,222 (27.54%)

adzuna_month06.npz:
  Original: 1,992,787
  Filtered: 1,287,469
  Reduction: 705,318 (35.39%)

adzuna_month07.npz:
  Original: 2,183,287
  Filtered: 1,484,251
  Reduction: 699,036 (32.02%)

adzuna_month08.npz:
  Original: 1,913,920
  Filtered: 1,296,837
  Reduction: 617,083 (32.24%)

adzuna_month09.npz:
  Original: 1,860,287
  Filtered: 1,269,039
  Reduction: 591,248 (31.78%)

adzuna_month10.npz:
  Original: 2,179,371
  Filtered: 1,486,281
  R

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import time
from datetime import datetime
import gc

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
t0 = time.time()

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")

out_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated")
out_root.mkdir(parents=True, exist_ok=True)

parquet_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet")

df = pd.read_parquet(parquet_file)
df["id"] = df["id"].astype(str)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

total_original = 0
total_filtered = 0

for month in months:
    npz_path = src_root / f"{month}.npz"
    out_path = out_root / f"{month}.npz"

    if not npz_path.exists():
        print(f"Skipping {month}: source file not found")
        continue

    keep_ids = df.loc[df["npz_file"] == f"{month}.npz", "id"].to_numpy(dtype=str)

    if keep_ids.size == 0:
        print(f"Skipping {month}: no IDs in dedup parquet")
        continue

    with np.load(npz_path, allow_pickle=True) as data:
        job_ids = data["job_ids"].astype(str)
        n_original = job_ids.shape[0]

        if keep_ids.size > n_original:
            # Not necessarily wrong, but a good smoke test
            print(f"Warning {month}: keep_ids ({keep_ids.size:,}) > job_ids ({n_original:,})")

        mask = np.isin(job_ids, keep_ids)
        n_filtered = int(mask.sum())

        if n_filtered == 0:
            raise RuntimeError(f"{month}: mask empty (no overlap between NPZ job_ids and dedup IDs)")

        filtered_data = {}
        for key in data.files:
            arr = data[key]
            if getattr(arr, "shape", None) is None or arr.shape[0] != n_original:
                raise ValueError(f"{month}: array '{key}' first-dim {getattr(arr,'shape',None)} != job_ids {n_original}")
            filtered_data[key] = arr[mask]

    np.savez_compressed(out_path, **filtered_data)

    total_original += n_original
    total_filtered += n_filtered

    reduction = n_original - n_filtered
    reduction_pct = reduction / n_original * 100

    del filtered_data, job_ids, mask, keep_ids
    gc.collect()

    print(f"{month}.npz")
    print(f"  Original : {n_original:,}")
    print(f"  Filtered : {n_filtered:,}")
    print(f"  Dropped  : {reduction:,} ({reduction_pct:.2f}%)")
    print(f"  Saved to : {out_path}\n")

dt = time.time() - t0
total_reduction = total_original - total_filtered
total_reduction_pct = total_reduction / total_original * 100 if total_original else 0

print("=" * 60)
print("TOTAL")
print(f"Original : {total_original:,}")
print(f"Filtered : {total_filtered:,}")
print(f"Dropped  : {total_reduction:,} ({total_reduction_pct:.2f}%)")
print(f"Duration : {dt:.2f}s ({dt/60:.2f}m)")
print(f"Finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Started at: 2026-02-17 22:20:48

adzuna_month01.npz
  Original : 2,657,586
  Filtered : 2,657,586
  Dropped  : 0 (0.00%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated/adzuna_month01.npz

adzuna_month02.npz
  Original : 2,229,436
  Filtered : 1,539,063
  Dropped  : 690,373 (30.97%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated/adzuna_month02.npz

adzuna_month03.npz
  Original : 2,303,805
  Filtered : 1,684,605
  Dropped  : 619,200 (26.88%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated/adzuna_month03.npz

adzuna_month04.npz
  Original : 2,331,704
  Filtered : 1,500,944
  Dropped  : 830,760 (35.63%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store

Post-hoc NPZ verification script

What it checks:

file exists

job_ids present

all arrays have same first dimension

row count matches deduplicated parquet for that month

no silent truncation

In [3]:



import numpy as np
import pandas as pd
from pathlib import Path

src_root = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/"
    "stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated"
)

parquet_file = Path(   
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/"
    "AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet"
)

df = pd.read_parquet(parquet_file)
df["id"] = df["id"].astype(str)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

errors = 0

for month in months:
    npz_path = src_root / f"{month}.npz"

    if not npz_path.exists():
        print(f"ERROR {month}: output NPZ missing")
        errors += 1
        continue

    expected_n = df.loc[
        df["npz_file"] == f"{month}.npz", "id"
    ].nunique()

    with np.load(npz_path, allow_pickle=True) as data:
        if "job_ids" not in data.files:
            print(f"ERROR {month}: job_ids missing")
            errors += 1
            continue

        job_ids = data["job_ids"].astype(str)
        n = job_ids.shape[0]

        if n != expected_n:
            print(
                f"ERROR {month}: row count mismatch "
                f"(npz={n:,}, parquet={expected_n:,})"
            )
            errors += 1

        for key in data.files:
            arr = data[key]
            if arr.shape[0] != n:
                print(
                    f"ERROR {month}: array '{key}' "
                    f"has shape {arr.shape}, expected {n}"
                )
                errors += 1

    print(f"OK {month}: {n:,} rows")

print("=" * 60)
if errors == 0:
    print("ALL CHECKS PASSED")
else:
    print(f"FAILED WITH {errors} ERRORS")


OK adzuna_month01: 2,657,586 rows
OK adzuna_month02: 1,539,063 rows
OK adzuna_month03: 1,684,605 rows
OK adzuna_month04: 1,500,944 rows
OK adzuna_month05: 1,845,262 rows
OK adzuna_month06: 1,287,469 rows
OK adzuna_month07: 1,484,251 rows
OK adzuna_month08: 1,296,837 rows
OK adzuna_month09: 1,269,039 rows
OK adzuna_month10: 1,486,281 rows
OK adzuna_month11: 1,419,300 rows
OK adzuna_month12: 982,172 rows
OK adzuna_month13: 1,220,262 rows
OK adzuna_month14: 8,426 rows
ALL CHECKS PASSED


In [ ]:
import numpy as np
from pathlib import Path
import time

print("--- STARTING FULL DATASET RUN ---")

# === PATH CONFIGURATION ===
src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated_FULL")

dst_root.mkdir(parents=True, exist_ok=True)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

def prepend_domain(text_arr, domain_arr):
    # This creates a new list in memory. For 1M+ rows, this is RAM heavy.
    out = []
    # We iterate directly over the mmap arrays to avoid loading source twice
    for t, d in zip(text_arr, domain_arr):
        t = "" if t is None else str(t).strip()
        d = "" if d is None else str(d).strip()
        out.append(f"[{d}] {t}" if d else t)
    return np.array(out, dtype=object)

for month in months:
    start_time = time.perf_counter()
    
    src_file = src_root / f"{month}.npz"
    dst_file = dst_root / f"{month}.npz"

    if not src_file.exists():
        print(f"Skipping {month}: Not found")
        continue

    # Load with mmap (Crucial for memory management)
    with np.load(src_file, allow_pickle=True, mmap_mode="r") as data:
        keys = list(data.files)
        
        # 1. Load everything as a reference first (Lazy loading where possible)
        # Note: We use data[k] directly, not data[k][:], to keep it as a file reference until needed
        sampled = {k: data[k] for k in keys} 
        
        n_total = len(data[keys[0]])

        # 2. MODIFY TEXT (This forces the text columns into RAM)
        if "domains" in sampled:
            # We access the mmap arrays directly in the function arguments
            if "role_text" in sampled:
                print(f"  Prepending domains to role_text ({n_total} rows)...")
                sampled["role_text"] = prepend_domain(sampled["role_text"], sampled["domains"])
            
            if "taskskill_text" in sampled:
                print(f"  Prepending domains to taskskill_text ({n_total} rows)...")
                sampled["taskskill_text"] = prepend_domain(sampled["taskskill_text"], sampled["domains"])

        # 3. SAVE FULL DATASET
        print(f"  Saving to {dst_file.name}...")
        np.savez_compressed(dst_file, **sampled)

    end_time = time.perf_counter()
    print(f"Processed {month}: {n_total} rows (FULL)")
    print(f"  Time: {end_time - start_time:.4f}s")
    print("-" * 40)